In [ ]:
from IPython.display import HTML
HTML(open('../style.css', 'r').read())

In [ ]:
from typing import TypeVar
from typing import Literal

# Computing the Intersection of Regular Expressions

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright 11-Intersection.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

Since *basedpyright* cannot follow the magic command `%run`, it reports the functions that are defined in the
notebooks loaded below as undefined.  We use `# type: ignore` to suppress these errors.

Regular expressions have no operator for the *logical and*, i.e. there is no operator $\wedge$ such that
$$ L(r_1 \wedge r_2) = L(r_1) \cap L(r_2). $$
Nevertheless, the regular languages are closed under intersection.  Hence, given two regular
expressions $r_1$ and $r_2$, there is a regular expression $r$ such that 
$L(r) = L(r_1) \cap L(r_2)$.  In order to compute $r$, we perform the following steps:
- convert the regular expressions $r_1$ and $r_2$ into <span style="font-variant:small-caps;">Dfa</span>s $F_1$ and $F_2$ such that
  $L(F_1) = L(r_1)$ and $L(F_2) = L(r_2)$,
- compute the *product automaton* $F_1 \times F_2$ that accepts the language $L(F_1) \cap L(F_2)$,
- minimize $F_1 \times F_2$ and remove all states from which no accepting state can be reached,
- convert the resulting <span style="font-variant:small-caps;">Dfa</span> back into a regular expression and simplify this regular expression.

The notebook `09-Equivalence.ipynb` contains the functions `cartesian_product` and `regexp2DFA`.
The latter function converts a regular expression into an equivalent <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
%run 09-Equivalence.ipynb

The notebook `07-Minimize.ipynb` contains the function `minimize` that minimizes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
%run 07-Minimize.ipynb

The notebook `05-DFA-2-RegExp.ipynb` contains the function `dfa_2_regexp` that converts a <span style="font-variant:small-caps;">Dfa</span> into an equivalent regular expression.

In [ ]:
%run 05-DFA-2-RegExp.ipynb

The notebook `Rewrite.ipynb` contains the function `simplify` that simplifies a regular expression.

In [ ]:
%run Rewrite.ipynb

Since the notebooks loaded above define types of the same name in different ways, we define the types used
in this notebook once more.

In [ ]:
type BinaryOp = Literal['⋅', '+']
type UnaryOp  = Literal['*']

In [ ]:
Char        = str
type RegExp = int | Char | tuple[RegExp, UnaryOp] | tuple[RegExp, BinaryOp, RegExp]
State       = TypeVar('State')
StatePair   = tuple[State, State]
TransRel1   = dict[tuple[State, Char], State]
TransRel2   = dict[tuple[StatePair, Char], StatePair]
DFA1        = tuple[set[State], set[Char], TransRel1, State, set[State]]
DFA2        = tuple[set[StatePair], set[Char], TransRel2, StatePair, set[StatePair]]

Given two <span style="font-variant:small-caps;">Dfa</span>s 
$$F_1 = \langle Q_1, \Sigma, \delta_1, q_1, A_1 \rangle \quad\mbox{and}\quad
  F_2 = \langle Q_2, \Sigma, \delta_2, q_2, A_2 \rangle,
$$
the *product automaton* $F_1 \times F_2$ is defined as
$$ F_1 \times F_2 := \bigl\langle Q_1 \times Q_2, \Sigma, \delta, \langle q_1, q_2 \rangle, A_1 \times A_2 \bigr\rangle
   \quad\mbox{where}\quad
   \delta\bigl(\langle p_1, p_2 \rangle, c\bigr) := \bigl\langle \delta_1(p_1, c), \delta_2(p_2, c) \bigr\rangle.
$$
The product automaton simulates $F_1$ and $F_2$ simultaneously.  It accepts a string $s$ iff both 
$F_1$ and $F_2$ accept $s$.  Therefore, we have
$$ L(F_1 \times F_2) = L(F_1) \cap L(F_2). $$
The only difference to the function `fsm_complement` in the notebook `09-Equivalence.ipynb` is the set of 
accepting states:  There, the set of accepting states is $A_1 \times (Q_2 \backslash A_2)$, here it is 
$A_1 \times A_2$.

**Function `fsm_intersection(F1, F2)`**
- *Input:* `F1` and `F2` are complete <span style="font-variant:small-caps;">Dfa</span>s with the same alphabet.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> that accepts the language $L(F_1) \cap L(F_2)$.

In [ ]:
def fsm_intersection(F1: DFA1, F2: DFA1) -> DFA2:
    States1, Σ, 𝛿1, q1, A1 = F1
    States2, _, 𝛿2, q2, A2 = F2
    States = cartesian_product(States1, States2)  # type: ignore
    𝛿: TransRel2 = {}
    for p1, p2 in States:
        for c in Σ:
            𝛿[(p1, p2), c] = (𝛿1[p1, c], 𝛿2[p2, c])
    return States, Σ, 𝛿, (q1, q2), cartesian_product(A1, A2)  # type: ignore

The function `dfa_2_regexp` computes the regular expressions $r_{p,q}$ recursively.  Every call of the
function `rpq` calls itself four times.  Hence, the running time of `dfa_2_regexp` grows exponentially
with the number of states.  Therefore, it is essential to keep the number of states small.  The function
`minimize` already removes all states that are not reachable from the start state and merges equivalent states.
However, the minimized <span style="font-variant:small-caps;">Dfa</span> can still contain a *trap state*, i.e. a state from which no accepting state can
be reached.  As `dfa_2_regexp` does not need a complete <span style="font-variant:small-caps;">Dfa</span>, we can remove this state.

Given a <span style="font-variant:small-caps;">Dfa</span> $F$, the function `productive(F)` computes the set of those states from which an accepting state
can be reached.  The variable `Result` is the set of those states that are already known to be productive.
Initially, these are the accepting states.  A state `p` is productive if there is a character `c` such that 
$\delta(p, c)$ is productive.

**Function `productive(F)`**
- *Input:* `F` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* The set of all states of `F` from which an accepting state can be reached.

In [ ]:
def productive(F: DFA1) -> set[State]:
    States, Σ, 𝛿, q0, Accepting = F
    Result = set(Accepting)
    while True:
        NewStates = { p for p in States for c in Σ if 𝛿[p, c] in Result }
        if NewStates <= Result:
            return Result
        Result |= NewStates

**Function `trim(F)`**
- *Input:* `F` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A partial <span style="font-variant:small-caps;">Dfa</span> that accepts the same language as `F` but that contains only productive states.
  If the start state of `F` is not productive, the language $L(F)$ is empty.  In this case, the result is `None`.

In [ ]:
def trim(F: DFA1) -> DFA1 | None:
    States, Σ, 𝛿, q0, Accepting = F
    Useful = productive(F)
    if q0 not in Useful:
        return None
    New𝛿: TransRel1 = { (p, c): q for (p, c), q in 𝛿.items() if p in Useful and q in Useful }
    return Useful, Σ, New𝛿, q0, Accepting & Useful

Given two regular expressions $r_1$ and $r_2$ and an alphabet $\Sigma$, the function 
`regexp_intersection(r1, r2, Σ)` computes a regular expression $r$ such that
$$ L(r) = L(r_1) \cap L(r_2). $$

**Function `regexp_intersection(r1, r2, Σ)`**
- *Input:* `r1` and `r2` are regular expressions and `Σ` is the alphabet used in them.
- *Output:* A regular expression `r` such that $L(r) = L(r_1) \cap L(r_2)$.

In [ ]:
def regexp_intersection(r1: RegExp, r2: RegExp, Σ: set[Char]) -> RegExp:
    F1 = regexp2DFA(r1, Σ)          # type: ignore
    F2 = regexp2DFA(r2, Σ)          # type: ignore
    F  = fsm_intersection(F1, F2)   # type: ignore
    M  = minimize(F)                # type: ignore
    T  = trim(M)                    # type: ignore
    if T is None:
        return 0
    r = dfa_2_regexp(T)             # type: ignore
    return simplify(r)              # type: ignore

The notebook `12-Test-Intersection.ipynb` can be used to test this function.